In [ ]:
"credit_risk_dataset_500_semi_prepared"# -*- coding: utf-8 -*-
import dataiku
import pandas as pd, numpy as np
from dataiku import pandasutils as pdu

# 1. קריאת הנתונים משלב הקלט
credit_risk_dataset_500_semi_prepared = dataiku.Dataset("credit_risk_dataset_500_semi_prepared")
# שמירת הנתונים למשתנה קצר (df) לעבודה נוחה יותר
df = credit_risk_dataset_500_semi_prepared.get_dataframe()

# ==========================================
# תחילת הלוגיקה העסקית - Feature Engineering
# ==========================================

# א. מדד לחץ פיננסי (Financial Stress Index)
df['financial_stress_index'] = (
    (df['previous_defaults'] * 2) +
    df['delinquency_12m'] +
    (df['loan_to_income_ratio'] * 10)
).round(2)
df['is_high_risk_profile'] = np.where(df['financial_stress_index'] > 5, 1, 0)

# ב. הכנסה פנויה חודשית (Monthly Disposable Income)
df['monthly_disposable_income'] = ((df['annual_income'] / 12) - df['estimated_monthly_payment']).round(2)
bins = [-np.inf, 1000, 3000, 7000, np.inf]
labels = ['critical', 'tight', 'comfortable', 'wealthy']
df['disposable_income_tier'] = pd.cut(df['monthly_disposable_income'], bins=bins, labels=labels)

# ג. דירוג אשראי וגיל סיום הלוואה (Credit Tier & Age Factor)
conditions = [
    (df['credit_score'] >= 800),
    (df['credit_score'] >= 740) & (df['credit_score'] < 800),
    (df['credit_score'] >= 670) & (df['credit_score'] < 740),
    (df['credit_score'] >= 580) & (df['credit_score'] < 670),
    (df['credit_score'] < 580)
]
choices = ['Excellent', 'Very Good', 'Good', 'Fair', 'Poor']
df['credit_score_tier'] = np.select(conditions, choices, default='Unknown')

df['age_at_loan_maturity'] = df['age'] + (df['loan_term_months'] / 12).round(2)

# ==========================================
# סיום הלוגיקה העסקית
# ==========================================

# 2. חיבור חזרה למשתנה הפלט של Dataiku
credit_risk_dataset_500_final_df = df

# 3. כתיבת התוצאות לשלב הבא ב-Flow
credit_risk_dataset_500_final = dataiku.Dataset("credit_risk_dataset_500_final")
credit_risk_dataset_500_final.write_with_schema(credit_risk_dataset_500_final_df)